# Errors and validation: handling bad input safely

Exceptions signal that something went wrong. try/except catches specific errors you can handle, else runs when nothing failed and finally always runs. In data pipelines, invalid records should be moved to a quarantine list with a reason instead of crashing the job or being silently dropped, and real failures should still stop the job.

**How to run in Google Colab:** run the cells from top to bottom (Runtime → Run all). Each part asks you to predict the result first; the expected results are listed at the end. For Databricks features such as Unity Catalog tables, use the Databricks Free Edition version of this lab.

From the Learning Hub lesson: [Errors and validation: handling bad input safely](https://www.datapreplab.com/hub#topic/py-errors-validation) · Independent study resource, not affiliated with or endorsed by Databricks.

**Goal:** see which blocks run, then build a validator with a quarantine.

**Part A: which blocks run?** Predict, then run with `raw = "12"` and again with `raw = "12a"`:

In [ ]:
def parse(raw):
    try:
        value = int(raw)
    except ValueError:
        print("except")
        return None
    else:
        print("else")
        return value
    finally:
        print("finally")

print(parse("12"))
print(parse("12a"))

**Part B: KeyError message.**

In [ ]:
d = {"a": 1}
try:
    d["b"]
except KeyError as e:
    print("missing", e)          # missing 'b'

**Part C: validator and quarantine.** Predict which rows are quarantined, then run:

In [ ]:
def validate(row):
    if not row.get("id"):
        return False, "missing id"
    try:
        amount = float(row["amount"])
    except (KeyError, ValueError):
        return False, "amount not numeric"
    if amount <= 0:
        return False, "amount must be positive"
    return True, {"id": int(row["id"]), "amount": amount}

rows = [{"id": "1", "amount": "120.5"}, {"id": "", "amount": "30"},
        {"id": "3", "amount": "n/a"}, {"id": "4", "amount": "-5"}]
valid, quarantine = [], []
for row in rows:
    ok, result = validate(row)
    if ok:
        valid.append(result)
    else:
        quarantine.append({**row, "reason": result})
print(valid)
print(quarantine)

**Part D: find the gap.** This row passes every check until `int(row["id"])`, which raises an error:

In [ ]:
try:
    validate({"id": "abc", "amount": "10"})
except ValueError as e:
    print("validate crashed:", e)

Add a rule to `validate` that quarantines it with the reason "id not numeric", then rerun.

Expected evidence:

```text
Part A -> else, finally, 12 ; except, finally, None
Part B -> missing 'b'
Part C -> valid [{'id': 1, 'amount': 120.5}] ; 3 quarantined with reasons
Part D -> ValueError before the fix ; quarantined with 'id not numeric' after
```